# Rotor-coupled optical scattering for `16O + 44Ca`

This example shows a heavier rotor-coupled optical model. You do **not** need the original paper to understand the notebook: think of it as a small recipe for building a multichannel optical potential from a reusable preset model, solving the scattering problem, and reading off the entrance-channel couplings.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

import lax as lm
from lax.boundary import BoundaryValues


def benchmark_data_dir() -> Path:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    search_roots.append(Path(lm.__file__).resolve().parents[2])
    for root in search_roots:
        candidate = root / "tests" / "benchmarks" / "data"
        if candidate.is_dir():
            return candidate
    msg = (
        "Could not locate tests/benchmarks/data from the current notebook environment."
    )
    raise FileNotFoundError(msg)


fixture = json.loads((benchmark_data_dir() / "descouvemont_o16_ca44.json").read_text())
reference = fixture["references"][2]

model = lm.models.O16_CA44_ROTOR_MODEL
channels = lm.models.channels_from_rotor_model(model)
energies = np.asarray(reference["energies"], dtype=np.float64)

[
    {
        "index": index,
        "label": channel.label,
        "threshold_mev": channel.threshold,
    }
    for index, channel in enumerate(model.channels)
]

## Why use the public preset?

The preset captures the physical bookkeeping that a new user should not have to re-type from memory:

- the channel list,
- the optical-model geometry,
- the Coulomb charges,
- and the deformation/coupling settings.

That makes the notebook a reusable template: copy the preset, change a few fields, and re-run the same workflow on a nearby problem.


In [ ]:
def boundary_at_energy(boundary: BoundaryValues, energy_index: int) -> BoundaryValues:
    return BoundaryValues(
        H_plus=boundary.H_plus[energy_index],
        H_minus=boundary.H_minus[energy_index],
        H_plus_p=boundary.H_plus_p[energy_index],
        H_minus_p=boundary.H_minus_p[energy_index],
        is_open=boundary.is_open[energy_index],
        k=boundary.k[energy_index],
    )


solver = lm.compile(
    mesh=lm.MeshSpec(
        "legendre",
        "x",
        n=int(reference["n_basis"]),
        scale=float(reference["scale"]),
        extras={"n_intervals": int(reference["n_intervals"])},
    ),
    channels=channels,
    operators=("T+L", "1/r^2"),
    solvers=("rmatrix_direct",),
    energies=energies,
    method="linear_solve",
    V_is_complex=True,
    z1z2=(model.projectile_charge, model.target_charge),
)

interaction = lm.models.interaction_from_rotor_model(model, solver)
r_values = solver.rmatrix_direct(interaction)

rows = []
for energy_index, energy in enumerate(energies):
    boundary = boundary_at_energy(solver.boundary, energy_index)
    smatrix = np.asarray(
        lm.spectral.open_channel_smatrix_from_R(r_values[energy_index], boundary)
    )
    open_count = lm.models.open_channel_count(model, float(energy))
    amplitudes, phases = lm.models.first_column_amplitudes_and_phases(
        smatrix, open_count
    )
    rows.append(
        {
            "energy_mev": float(energy),
            "open_channels": open_count,
            "entrance_column_amplitudes": amplitudes.tolist(),
            "reference_amplitudes": reference["amplitudes"][energy_index],
            "entrance_column_phases": phases.tolist(),
            "reference_phases": reference["phases"][energy_index],
        }
    )

rows

## Interpreting the output

Each row corresponds to one beam energy. The `entrance_column_amplitudes` list tells you how strongly the incoming elastic channel couples into each open outgoing channel. The matching `entrance_column_phases` list tells you the phase shift attached to those outgoing amplitudes.

In many exploratory studies that is exactly the right level of summary: enough to see whether a coupling is weak, strong, opening, or changing sign, without digging into the full matrix immediately.
